# Imports and configuration

The notebook uses:

- **PyTorch** to create datasets, load batches, train the model, and export the trained model into the ONNX format.
- **Hugging Face Transformers** to load the pretrained TinyBERT model and its tokenizer.
- **PEFT** to fine-tune only a small number of additional parameters using LoRA.


## Note

For running this notebook, create a new project using the *ECAI Template* because the template has been updated adding new dependencies and files.

In [5]:
import json
import os
import random
import time
from datetime import datetime
from typing import Any

import numpy as np
import torch
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Configuration Parameters

In [7]:
MODEL_NAME = "huawei-noah/TinyBERT_General_4L_312D"
TRAIN_FILE = "train.jsonl"
TEST_FILE = "test.jsonl"
OUTPUT_DIR = "results"
EPOCHS = 20
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
MAX_LENGTH = 64
SEED = 42
SKIP_ONNX = False
LABELS = [
    "enable_cloud",
    "disable_cloud",
    "increase_interval",
    "decrease_interval",
    "unknown",
]
LABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# The dataset class

`IntentDataset` reads the JSONL file one line at a time. Each line contains a text sentence and its intent label, for example:

```json
{"text": "Start sending measurements to cloud storage.", "label": "enable_cloud"}
```

The class stores the sentences and converts their labels into integer IDs. The mapping is defined by `LABEL_TO_ID` at the beginning of the script. A neural network cannot directly use strings such as `enable_cloud` as target values, so each class is represented by an integer.

The class implements two methods:

- `__len__()` returns the number of examples in the dataset.
- `__getitem__(index)` returns one tokenized sentence and its label. This is the interface used by PyTorch's `DataLoader`.

## Tokenization

A Transformer does not receive raw text. It receives a sequence of token IDs. The tokenizer splits each sentence into tokens (usually subword units) and converts them into integer IDs understood by transformers.

The tokenizer is called with the following options:

```python
tokenizer(
    texts,
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH,
    return_tensors="pt",
)
```

These options have the following meaning:

- `truncation=True`: sentences longer than 64 tokens are cut to the maximum length.
- `padding="max_length"`: shorter sentences are padded so that every example has exactly 64 tokens.
- `return_tensors="pt"`: the tokenizer returns PyTorch tensors.

The tokenizer produces three inputs:

- `input_ids`: the numerical representation of the sentence.
- `attention_mask`: indicates which positions contain real tokens and which positions are padding. The model should ignore the padding positions.
- `token_type_ids`: identifies the sentence segment. For these single-sentence commands, the examples use one segment.

Padding makes it possible to combine sentences of different lengths into the same batch. Truncation ensures that no example exceeds the configured maximum length.


In [9]:
class IntentDataset(Dataset):
    def __init__(
        self,
        path: str,
        tokenizer: Any,
        label2id: dict[str, int],
        max_length: int,
    ):
        texts = []
        labels = []
        with open(path, "r") as fp:
            for line in fp:
                row = json.loads(line)
                texts.append(row["text"])
                labels.append(row["label"])

        tokenized_data = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        self.input_ids = tokenized_data["input_ids"]
        self.attention_mask = tokenized_data["attention_mask"]
        self.token_type_ids = tokenized_data["token_type_ids"]
        self.labels = torch.tensor(
            [label2id[label] for label in labels], dtype=torch.long
        )

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        return {
            "input_ids": self.input_ids[index],
            "attention_mask": self.attention_mask[index],
            "token_type_ids": self.token_type_ids[index],
            "labels": self.labels[index],
        }

# DataLoaders

We create one `DataLoader` for the training set and one for the test set. A `DataLoader` groups individual examples into batches and provides them to the training loop.

The training loader uses `shuffle=True`, so the examples are presented in a different order during each epoch. The test loader uses `shuffle=False`, because it is only used to evaluate the model.



In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_data = IntentDataset(TRAIN_FILE, tokenizer, LABEL_TO_ID, MAX_LENGTH)
test_data = IntentDataset(TEST_FILE, tokenizer, LABEL_TO_ID, MAX_LENGTH)
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_data, batch_size=BATCH_SIZE, shuffle=True, generator=generator
)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

# Model initialization

The model is loaded with `AutoModelForSequenceClassification`. This Hugging Face class adds a classification head on top of the pretrained Transformer. The head produces five output scores, one for each intent class.

The pretrained model was not originally trained for these five intents, so the classification head is initialized for the new task. The script then applies **LoRA** (Low-Rank Adaptation) using the PEFT library.

Instead of updating all the weights in the modeal, LoRA adds small trainable matrices to selected attention layers. The classification head is also kept trainable. Most of the pretrained model parameters remain frozen.

This approach reduces the number of parameters that need to be updated during training. 

In [13]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=dict(enumerate(LABELS)),
    label2id=LABEL_TO_ID,
    ignore_mismatched_sizes=True,
)
model = get_peft_model(
    base_model,
    LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=2,
        lora_alpha=4,
        lora_dropout=0.05,
        target_modules=["query", "value"],
        modules_to_save=["classifier"],
    ),
)
trainable = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
total = sum(parameter.numel() for parameter in model.parameters())
print("Model parameter statistics:")
print(f"  Total parameters:     {total:,}")
print(f"  Trainable parameters: {trainable:,}")
print(f"  Frozen parameters:    {total - trainable:,}")
print(f"  Trainable fraction:   {trainable / total:.1%}", flush=True)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Model parameter statistics:
  Total parameters:     14,363,362
  Trainable parameters: 11,549
  Frozen parameters:    14,351,813
  Trainable fraction:   0.1%


# Optimizer

The script uses the AdamW optimizer. It assigns a learning rate of `0.001` to the LoRA parameters and a learning rate ten times larger (`0.01`) to the classification head. The head is newly initialized for this task, while the LoRA parameters adapt the pretrained representation.


In [15]:
print(model)

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): BertForSequenceClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 312, padding_idx=0)
          (position_embeddings): Embedding(512, 312)
          (token_type_embeddings): Embedding(2, 312)
          (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-3): 4 x BertLayer(
              (attention): BertAttention(
                (self): BertSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=312, out_features=312, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default)

In [19]:
classifier_parameters = [
    parameter
    for name, parameter in model.named_parameters()
    if parameter.requires_grad and "classifier" in name
]
classifier_ids = {id(parameter) for parameter in classifier_parameters}
adapter_parameters = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad and id(parameter) not in classifier_ids
]
optimizer = torch.optim.AdamW(
    [
        {"params": adapter_parameters, "lr": LEARNING_RATE},
        {"params": classifier_parameters, "lr": LEARNING_RATE * 10},
    ]
)

# Training loop

The model is trained for 20 epochs. During each epoch, the script:

1. Calls `model.train()` to enable training behavior.
2. Reads one batch from the training `DataLoader` and moves its tensors to the selected device (GPU when available, otherwise CPU).
3. Runs the model to obtain the output scores, called **logits**.
4. Computes the cross-entropy loss between the logits and the correct intent labels.
5. Calls `loss.backward()` to calculate gradients and `optimizer.step()` to update the trainable parameters.

`optimizer.zero_grad()` clears the gradients before the next update. This is necessary because PyTorch accumulates gradients by default.

At the end of every epoch, the script evaluates the model on the test set. It calls `model.eval()` and uses `torch.inference_mode()` so that gradients are not calculated during evaluation.

The script computes the average test loss and classification accuracy. Accuracy is the percentage of test sentences for which the predicted class is equal to the correct label. The training loss, test loss, and test accuracy are printed after every epoch.

In [21]:
epoch_metrics = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum = 0.0
    seen = 0
    for batch in train_loader:
        labels = batch.pop("labels")
        optimizer.zero_grad(set_to_none=True)
        logits = model(**batch).logits
        loss = nn.functional.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * labels.numel()
        seen += labels.numel()

    model.eval()
    correct = 0
    examples = 0
    test_loss_sum = 0.0
    with torch.inference_mode():
        for batch in test_loader:
            labels = batch.pop("labels")
            logits = model(**batch).logits
            test_loss_sum += nn.functional.cross_entropy(
                logits, labels, reduction="sum"
            ).item()
            correct += (logits.argmax(dim=-1) == labels).sum().item()
            examples += labels.numel()
    metrics = {
        "epoch": epoch,
        "train_loss": loss_sum / seen,
        "test_loss": test_loss_sum / examples,
        "test_accuracy": 100 * correct / examples,
    }
    epoch_metrics.append(metrics)
    dt = datetime.now().strftime("%d/%m/%y %H:%M")
    print(
        f"{dt} - "
        f"Epoch {epoch:>2}/{EPOCHS} | "
        f"train loss: {metrics['train_loss']:.4f} | "
        f"test loss: {metrics['test_loss']:.4f} | "
        f"test accuracy: {metrics['test_accuracy']:.2f}%",
        flush=True,
    )

19/11/26 16:01 - Epoch  1/20 | train loss: 1.6057 | test loss: 1.5206 | test accuracy: 31.00%
19/11/26 16:01 - Epoch  2/20 | train loss: 1.2856 | test loss: 0.9731 | test accuracy: 65.00%
19/11/26 16:01 - Epoch  3/20 | train loss: 0.8527 | test loss: 0.7183 | test accuracy: 74.00%
19/11/26 16:02 - Epoch  4/20 | train loss: 0.6383 | test loss: 0.5345 | test accuracy: 77.50%
19/11/26 16:02 - Epoch  5/20 | train loss: 0.4725 | test loss: 0.4038 | test accuracy: 83.00%
19/11/26 16:03 - Epoch  6/20 | train loss: 0.3641 | test loss: 0.3190 | test accuracy: 88.00%
19/11/26 16:03 - Epoch  7/20 | train loss: 0.3219 | test loss: 0.2490 | test accuracy: 93.00%
19/11/26 16:03 - Epoch  8/20 | train loss: 0.2590 | test loss: 0.2145 | test accuracy: 93.00%
19/11/26 16:04 - Epoch  9/20 | train loss: 0.2131 | test loss: 0.1787 | test accuracy: 94.50%
19/11/26 16:04 - Epoch 10/20 | train loss: 0.1902 | test loss: 0.1406 | test accuracy: 97.50%
19/11/26 16:05 - Epoch 11/20 | train loss: 0.1411 | test los

# Exporting the model to ONNX

After training, the script saves the LoRA adapter and exports the trained classifier to `results/intent_classifier.onnx`.

ONNX is a portable model format. It allows the Raspberry Pi to run the classifier with ONNX Runtime, without installing PyTorch or Hugging Face Transformers for inference.

The export wrapper exposes a simple interface: it receives `input_ids` and `attention_mask`, and returns the model's logits. The export also declares the batch size and sequence length as dynamic dimensions, so the model is not restricted to only one fixed input shape.


In [23]:
class LogitsOnly(nn.Module):
    """Keep the ONNX graph's public interface to just IDs, mask, and logits."""

    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def forward(
        self, input_ids: torch.Tensor, attention_mask: torch.Tensor
    ) -> torch.Tensor:
        return self.model(input_ids=input_ids, attention_mask=attention_mask).logits

onnx_path = os.path.join(OUTPUT_DIR, "intent_classifier.onnx")
exporter = LogitsOnly(model).eval()
encoded = tokenizer(
    "export validation sample",
    return_tensors="pt",
    max_length=MAX_LENGTH,
    truncation=True,
)
input_ids = encoded["input_ids"]
attention_mask = encoded["attention_mask"]
torch.onnx.export(
    exporter,
    (input_ids, attention_mask),
    str(onnx_path),
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size"},
    },
    opset_version=17,
    dynamo=False,
)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save inference metadata
with open(
    os.path.join(OUTPUT_DIR, "labels.json"), "w", encoding="utf-8"
) as labels_file:
    labels_file.write(json.dumps(LABELS, indent=2))



/usr/local/lib/python3.13/site-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


# Log Training metadata

In [ ]:
# Write run metrics
result = {
    "model_name": MODEL_NAME,
    "strategy": "tiny_lora",
    "labels": LABELS,
    "train_samples": len(train_data),
    "test_samples": len(test_data),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "classifier_learning_rate": LEARNING_RATE * 10,
    "max_length": MAX_LENGTH,
    "seed": SEED,
    "final_test_accuracy": epoch_metrics[-1]["test_accuracy"],
    "final_test_loss": epoch_metrics[-1]["test_loss"],
    "epoch_metrics": epoch_metrics,
    "onnx_path": str(onnx_path) if not SKIP_ONNX else None,
}
with open(
    os.path.join(OUTPUT_DIR, "results.json"), "w", encoding="utf-8"
) as results_file:
    results_file.write(json.dumps(result, indent=2))
